In [1]:
## 2026 MONACO GRAND PRIX — PODIUM PREDICTION
### Rolling data: Australia (R1) + China (R2) + Japan (R3) + Miami (R4) + Canada (R5)
### Monaco GP qualifying grid sourced live (June 6 2026)
### Weather: dry, ~23°C race day, no rain expected

# ── Imports ───────────────────────────────────────────────────────────────────
import sys
!{sys.executable} -m pip install fastf1 --quiet

import fastf1
import pandas as pd
import numpy as np
from sklearn.linear_model import Ridge, ElasticNet
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import LeaveOneOut
from sklearn.impute import SimpleImputer
from xgboost import XGBRegressor

# ── Monaco GP qualifying grid (confirmed, June 6 2026) ────────────────────────
# Source: formula1.com, the-race.com, total-motorsport.com, crash.net
# ANT pole: 1:12.051 | VER +0.043 | HAM +0.228 | LEC +0.394 (estimated gap to pole)
# RUS P6 +0.394 (0.394s behind ANT per ESPN)
# NOTE: Leclerc hit wall on final lap in Q3 — car damage TBC for race start

monaco_quali_data = [
    # Driver, BestLap_s (ANT pole = 72.051), MonacoGrid
    # Gaps confirmed: VER +0.043, HAM +0.228, LEC TBC ~+0.3, HAD ~+0.35, RUS +0.394
    # PIA ~+0.5, NOR ~+0.55, GAS ~+0.7, LAW ~+0.8 (Q3/Q2 estimates)
    # P11-P22: using FP2/Q1-Q2 pace ordering with estimated gaps
    {"Driver": "ANT", "BestLap_s": 72.051, "MonacoGrid": 1},
    {"Driver": "VER", "BestLap_s": 72.094, "MonacoGrid": 2},
    {"Driver": "HAM", "BestLap_s": 72.279, "MonacoGrid": 3},
    {"Driver": "LEC", "BestLap_s": 72.380, "MonacoGrid": 4},
    {"Driver": "HAD", "BestLap_s": 72.420, "MonacoGrid": 5},
    {"Driver": "RUS", "BestLap_s": 72.445, "MonacoGrid": 6},
    {"Driver": "PIA", "BestLap_s": 72.560, "MonacoGrid": 7},
    {"Driver": "NOR", "BestLap_s": 72.620, "MonacoGrid": 8},
    {"Driver": "GAS", "BestLap_s": 72.780, "MonacoGrid": 9},
    {"Driver": "LAW", "BestLap_s": 72.900, "MonacoGrid": 10},
    {"Driver": "ALB", "BestLap_s": 73.100, "MonacoGrid": 11},
    {"Driver": "SAI", "BestLap_s": 73.150, "MonacoGrid": 12},
    {"Driver": "HUL", "BestLap_s": 73.200, "MonacoGrid": 13},
    {"Driver": "COL", "BestLap_s": 73.280, "MonacoGrid": 14},
    {"Driver": "LIN", "BestLap_s": 73.360, "MonacoGrid": 15},
    {"Driver": "BOR", "BestLap_s": 73.450, "MonacoGrid": 16},
    {"Driver": "OCO", "BestLap_s": 73.550, "MonacoGrid": 17},
    {"Driver": "PER", "BestLap_s": 73.700, "MonacoGrid": 18},
    {"Driver": "BEA", "BestLap_s": 73.800, "MonacoGrid": 19},
    {"Driver": "BOT", "BestLap_s": 73.950, "MonacoGrid": 20},
    {"Driver": "ALO", "BestLap_s": 74.100, "MonacoGrid": 21},
    {"Driver": "STR", "BestLap_s": 74.300, "MonacoGrid": 22},
]
# NOTE: Q1 eliminated (P17-P22): OCO, PER, BEA, BOT, ALO, STR per the-race.com
# NOTE: Q2 eliminated (P11-P16): ALB, SAI, HUL, COL, LIN, BOR per the-race.com

quali_df = pd.DataFrame(monaco_quali_data)

pole_time = quali_df["BestLap_s"].min()
quali_df["MonacoGapFromPole_s"]  = (quali_df["BestLap_s"] - pole_time).round(3)
quali_df["MonacoGapFromPolePct"] = (quali_df["MonacoGapFromPole_s"] / pole_time * 100).round(4)

# ── Race-day weather (Monaco, June 7 2026) ────────────────────────────────────
# Source: f1-fansite.com (qualifying conditions: dry, 23°C, 33°C tarmac)
# Race day forecast: sunny, warm, no rain expected — classic Monaco conditions
rain_probability = 0.05   # effectively dry
temperature      = 23.0   # ~23°C race day air temp

quali_df["RainProbability"] = rain_probability
quali_df["Temperature"]     = temperature

# ── Driver standings after Canadian GP (Round 5) ──────────────────────────────
# Sources: total-motorsport.com, si.com, crash.net (standings after Canada R5)
# ANT: 131, RUS: 88, LEC: 75, HAM: 72, NOR: 58, PIA: 49, VER: 47
# COL: 15, LAW: 13, GAS: 12, HAD: 11, BEA: 10, LIN: 8, SAI: 7
# ALB: 5, BOR: 2, OCO: 1, ALO/STR/PER/BOT/HUL: 0
# Note: HAD +7 pts from Canada P5; COL +8 pts from Canada P6 (best result)
driver_points = {
    "ANT": 131, "RUS": 88,  "LEC": 75,  "HAM": 72,
    "NOR": 58,  "PIA": 49,  "VER": 47,  "COL": 15,
    "LAW": 13,  "GAS": 12,  "HAD": 11,  "BEA": 10,
    "LIN": 8,   "SAI": 7,   "ALB": 5,   "BOR": 2,
    "OCO": 1,   "ALO": 0,   "STR": 0,   "PER": 0,
    "BOT": 0,   "HUL": 0,
}

max_driver_pts = max(v for v in driver_points.values() if v > 0)
quali_df["DriverPoints"]     = quali_df["Driver"].map(driver_points).fillna(0)
quali_df["DriverPointsNorm"] = (quali_df["DriverPoints"] / max_driver_pts).round(4)

print(quali_df[["Driver", "BestLap_s", "MonacoGrid", "MonacoGapFromPole_s",
                "MonacoGapFromPolePct", "RainProbability", "Temperature",
                "DriverPointsNorm"]])
quali_df.to_csv("monaco_quali.csv", index=False)
print("\nSaved → monaco_quali.csv")


# ── Rolling features: AUS + CHN + JPN + MIA + CAN (5 races) ──────────────────

# Finish positions
aus_finish = {
    "RUS": 1,  "ANT": 2,  "LEC": 3,  "HAM": 4,
    "NOR": 5,  "VER": 6,  "BEA": 7,  "LIN": 8,
    "BOR": 9,  "GAS": 10, "OCO": 11, "ALB": 12,
    "LAW": 13, "COL": 14, "SAI": 15, "PER": 16,
    "STR": 17, "ALO": 18, "BOT": 19, "HAD": 20,
    "PIA": 21, "HUL": 22,
}

china_finish = {
    "ANT": 1,  "RUS": 2,  "HAM": 3,  "LEC": 4,
    "BEA": 5,  "GAS": 6,  "LAW": 7,  "HAD": 8,
    "SAI": 9,  "COL": 10, "HUL": 11, "LIN": 12,
    "BOT": 13, "OCO": 14, "PER": 15, "VER": 16,
    "ALO": 17, "STR": 18, "NOR": 19, "BOR": 20,
    "ALB": 21, "PIA": 22,
}

japan_finish = {
    "ANT": 1,  "PIA": 2,  "LEC": 3,  "RUS": 4,
    "NOR": 5,  "HAM": 6,  "GAS": 7,  "VER": 8,
    "LAW": 9,  "OCO": 10, "HUL": 11, "HAD": 12,
    "BOR": 13, "LIN": 14, "SAI": 15, "COL": 16,
    "PER": 17, "ALO": 18, "BOT": 19, "ALB": 20,
    "STR": 21, "BEA": 22,
}

miami_finish = {
    "ANT": 1,  "NOR": 2,  "PIA": 3,  "RUS": 4,
    "VER": 5,  "HAM": 6,  "COL": 7,  "LEC": 8,
    "SAI": 9,  "ALB": 10, "BEA": 11, "BOR": 12,
    "OCO": 13, "LIN": 14, "ALO": 15, "PER": 16,
    "STR": 17, "BOT": 18, "HUL": 19, "LAW": 20,
    "GAS": 21, "HAD": 22,
}

# Canadian GP final classification (May 24 2026)
# Source: total-motorsport.com, scuderiafans.com, formula1archive.com
# RUS DNF (power unit); NOR DNF (reliability); many retirements
# P11 onwards: Piastri, Hulkenberg, Bortoleto, Ocon, Stroll, Bottas, + DNFs
canada_finish = {
    "ANT": 1,  "HAM": 2,  "VER": 3,  "LEC": 4,
    "HAD": 5,  "COL": 6,  "LAW": 7,  "GAS": 8,
    "SAI": 9,  "BEA": 10, "PIA": 11, "HUL": 12,
    "BOR": 13, "OCO": 14, "STR": 15, "BOT": 16,
    "ALB": 17, "ALO": 18, "LIN": 19, "NOR": 20,
    "RUS": 21, "PER": 22,  # RUS/NOR DNF → last; PER also DNF
}

# Grid/qualifying positions
aus_grid = {
    "RUS": 1,  "ANT": 2,  "HAD": 3,  "LEC": 4,  "PIA": 5,
    "NOR": 6,  "HAM": 7,  "LAW": 8,  "LIN": 9,  "BOR": 10,
    "OCO": 11, "HUL": 12, "ALB": 13, "GAS": 14, "COL": 15,
    "BEA": 16, "ALO": 17, "PER": 18, "BOT": 19, "VER": 20,
    "SAI": 21, "STR": 22,
}

china_grid = {
    "ANT": 1,  "RUS": 2,  "HAM": 3,  "LEC": 4,  "PIA": 5,
    "NOR": 6,  "GAS": 7,  "VER": 8,  "HAD": 9,  "BEA": 10,
    "HUL": 11, "COL": 12, "OCO": 13, "LAW": 14, "LIN": 15,
    "BOR": 16, "SAI": 17, "ALB": 18, "ALO": 19, "BOT": 20,
    "STR": 21, "PER": 22,
}

japan_grid = {
    "ANT": 1,  "RUS": 2,  "PIA": 3,  "LEC": 4,  "NOR": 5,
    "HAM": 6,  "GAS": 7,  "HAD": 8,  "BOR": 9,  "LIN": 10,
    "VER": 11, "OCO": 12, "HUL": 13, "LAW": 14, "COL": 15,
    "SAI": 16, "ALB": 17, "BEA": 18, "PER": 19, "BOT": 20,
    "ALO": 21, "STR": 22,
}

miami_grid = {
    "ANT": 1,  "NOR": 2,  "VER": 3,  "LEC": 4,  "PIA": 5,
    "RUS": 6,  "GAS": 7,  "BEA": 8,  "SAI": 9,  "OCO": 10,
    "HUL": 11, "LAW": 12, "LIN": 13, "ALB": 14, "BOR": 15,
    "COL": 16, "ALO": 17, "STR": 18, "BOT": 19, "HAM": 20,
    "HAD": 21, "PER": 22,
}

# Canadian GP qualifying grid (May 23 2026)
# Source: the-race.com, crash.net, gpfans.com
# STR started from pit lane (ESME replacement) → treated as P22
canada_grid = {
    "RUS": 1,  "ANT": 2,  "NOR": 3,  "PIA": 4,  "HAM": 5,
    "VER": 6,  "HAD": 7,  "LEC": 8,  "LIN": 9,  "COL": 10,
    "HUL": 11, "LAW": 12, "BOR": 13, "GAS": 14, "SAI": 15,
    "BEA": 16, "OCO": 17, "ALB": 18, "ALO": 19, "PER": 20,
    "BOT": 21, "STR": 22,
}

# Monaco GP qualifying grid (June 6 2026)
monaco_grid = {
    "ANT": 1,  "VER": 2,  "HAM": 3,  "LEC": 4,  "HAD": 5,
    "RUS": 6,  "PIA": 7,  "NOR": 8,  "GAS": 9,  "LAW": 10,
    "ALB": 11, "SAI": 12, "HUL": 13, "COL": 14, "LIN": 15,
    "BOR": 16, "OCO": 17, "PER": 18, "BEA": 19, "BOT": 20,
    "ALO": 21, "STR": 22,
}

# Constructor standings after Canadian GP (Round 5)
# Source: si.com, total-motorsport.com, crash.net
# MER: 219, FER: 147, MCL: 107 (NOR DNF Canada), RBR: 58, ALP: 27
# RBU: 21, HAA: 11, WIL: 7, AUD: 2, CAD: 0, AST: 0
team_points = {
    "Mercedes":     219,
    "Ferrari":      147,
    "McLaren":      107,
    "Red Bull":      58,
    "Alpine":        27,
    "Racing Bulls":  21,
    "Haas":          11,
    "Williams":       7,
    "Audi":           2,
    "Cadillac":       0,
    "Aston Martin":   0,
}

driver_to_team = {
    "RUS": "Mercedes",     "ANT": "Mercedes",
    "HAM": "Ferrari",      "LEC": "Ferrari",
    "NOR": "McLaren",      "PIA": "McLaren",
    "VER": "Red Bull",     "HAD": "Red Bull",
    "BEA": "Haas",         "OCO": "Haas",
    "LAW": "Racing Bulls", "LIN": "Racing Bulls",
    "HUL": "Audi",         "BOR": "Audi",
    "GAS": "Alpine",       "COL": "Alpine",
    "SAI": "Williams",     "ALB": "Williams",
    "PER": "Cadillac",     "BOT": "Cadillac",
    "ALO": "Aston Martin", "STR": "Aston Martin",
}

# ── Build 5-race rolling features ─────────────────────────────────────────────
drivers = list(driver_to_team.keys())
N = len(drivers)  # 22

records = []
for drv in drivers:
    aus_fp  = aus_finish[drv];    chn_fp = china_finish[drv]
    jpn_fp  = japan_finish[drv];  mia_fp = miami_finish[drv]
    can_fp  = canada_finish[drv]
    aus_gp  = aus_grid[drv];      chn_gp = china_grid[drv]
    jpn_gp  = japan_grid[drv];    mia_gp = miami_grid[drv]
    can_gp  = canada_grid[drv]

    avg_finish_pos  = round((aus_fp + chn_fp + jpn_fp + mia_fp + can_fp) / 5, 2)
    avg_finish_norm = round(
        ((aus_fp-1)/(N-1) + (chn_fp-1)/(N-1) + (jpn_fp-1)/(N-1)
         + (mia_fp-1)/(N-1) + (can_fp-1)/(N-1)) / 5, 4
    )

    avg_grid_pos  = round((aus_gp + chn_gp + jpn_gp + mia_gp + can_gp) / 5, 2)
    avg_grid_norm = round(
        ((aus_gp-1)/(N-1) + (chn_gp-1)/(N-1) + (jpn_gp-1)/(N-1)
         + (mia_gp-1)/(N-1) + (can_gp-1)/(N-1)) / 5, 4
    )

    # DNF flags
    dnf_aus = 1 if aus_fp >= 18 else 0
    dnf_chn = 1 if chn_fp >= 16 else 0
    dnf_jpn = 1 if jpn_fp >= 21 else 0
    dnf_mia = 1 if mia_fp >= 19 else 0
    dnf_can = 1 if can_fp >= 20 else 0   # RUS/NOR DNF → P20/P21
    dnf_rate = round((dnf_aus + dnf_chn + dnf_jpn + dnf_mia + dnf_can) / 5, 3)

    # Recent form: exponential decay (CAN 40%, MIA 25%, JPN 20%, CHN 10%, AUS 5%)
    recent_finish_norm = round(
        0.40*(can_fp-1)/(N-1) + 0.25*(mia_fp-1)/(N-1) +
        0.20*(jpn_fp-1)/(N-1) + 0.10*(chn_fp-1)/(N-1) +
        0.05*(aus_fp-1)/(N-1), 4
    )

    records.append({
        "Driver":            drv,
        "Team":              driver_to_team[drv],
        "AusFinish":         aus_fp, "ChinaFinish": chn_fp,
        "JapanFinish":       jpn_fp, "MiamiFinish": mia_fp,
        "CanadaFinish":      can_fp,
        "AvgFinishPos":      avg_finish_pos,
        "AvgFinishNorm":     avg_finish_norm,
        "RecentFinishNorm":  recent_finish_norm,
        "AusGrid":           aus_gp, "ChinaGrid":   chn_gp,
        "JapanGrid":         jpn_gp, "MiamiGrid":   mia_gp,
        "CanadaGrid_prev":   can_gp,
        "AvgGridPos":        avg_grid_pos,
        "AvgGridNorm":       avg_grid_norm,
        "DNF_rate":          dnf_rate,
    })

rolling_df = pd.DataFrame(records)

max_team_pts   = max(team_points.values())
max_driver_pts = max(driver_points.values())

rolling_df["TeamScore"] = rolling_df["Team"].map(
    {t: max(p, 0.5) / max_team_pts for t, p in team_points.items()}
).round(4)

rolling_df["DriverPointsNorm"] = rolling_df["Driver"].map(
    {d: p / max_driver_pts for d, p in driver_points.items()}
).round(4)

# Print rolling features summary
print("\n" + "=" * 115)
print("  ROLLING FEATURES FOR MONACO GP PREDICTION")
print("  Based on: Australia (R1) + China (R2) + Japan (R3) + Miami (R4) + Canada (R5)")
print("=" * 115)
print(f"\n{'DRV':<6}{'Team':<16}{'AUS':<5}{'CHN':<5}{'JPN':<5}{'MIA':<5}{'CAN':<5}{'AvgFin':<9}"
      f"{'RecentNorm':<12}{'AvgGrid':<9}{'AvgGridNorm':<13}"
      f"{'DNF%':<7}{'TeamScore':<11}{'DrvPtsNorm'}")
print("─" * 120)
for _, r in rolling_df.sort_values("AvgFinishPos").iterrows():
    print(f"{r['Driver']:<6}{r['Team']:<16}{r['AusFinish']:<5}{r['ChinaFinish']:<5}"
          f"{r['JapanFinish']:<5}{r['MiamiFinish']:<5}{r['CanadaFinish']:<5}{r['AvgFinishPos']:<9}"
          f"{r['RecentFinishNorm']:<12}{r['AvgGridPos']:<9}{r['AvgGridNorm']:<13}"
          f"{r['DNF_rate']:<7}{r['TeamScore']:<11}{r['DriverPointsNorm']}")

rolling_df.to_csv("rolling_features_aus_chn_jpn_mia_can.csv", index=False)
print("\nSaved → rolling_features_aus_chn_jpn_mia_can.csv")


# ── Load FP2 race pace (Monaco has FP1 + FP2 + FP3 — no sprint) ──────────────
# Using FastF1 to fetch FP2 lap times as long-run pace proxy
# FP2 top result: HAM 1:13.026, LEC +0.111, VER +0.168, RUS ~+0.3, ANT ~+1.0s
# Source: formula1.com, racingnews365.com, total-motorsport.com

def td_to_s(td):
    if pd.isnull(td):
        return np.nan
    return td.total_seconds()

try:
    session_fp2 = fastf1.get_session(2026, "Monaco", "FP2")
    session_fp2.load(telemetry=False, weather=False, messages=False)

    fp2_laps = session_fp2.laps.copy()
    fp2_laps["LapTime_s"] = fp2_laps["LapTime"].apply(td_to_s)
    fp2_laps = fp2_laps[fp2_laps["IsAccurate"] == True]

    def within_107(group):
        med = group["LapTime_s"].median()
        return group[group["LapTime_s"] <= med * 1.07]

    fp2_laps = fp2_laps.groupby("Driver", group_keys=False).apply(within_107)

    fp2_records = []
    for driver, grp in fp2_laps.groupby("Driver"):
        fp2_records.append({
            "Driver":           driver,
            "FP2_MedianPace_s": round(grp["LapTime_s"].median(), 3),
            "FP2_LapCount":     len(grp),
            "FP2_Compounds":    str(grp["Compound"].value_counts().to_dict()),
        })

    fp2_df = pd.DataFrame(fp2_records).sort_values("FP2_MedianPace_s").reset_index(drop=True)
    ref_pace = fp2_df["FP2_MedianPace_s"].min()
    fp2_df["FP2_GapToFastest_s"] = (fp2_df["FP2_MedianPace_s"] - ref_pace).round(3)
    print("\nFP2 race pace (long-run proxy):")
    print(fp2_df[["Driver", "FP2_MedianPace_s", "FP2_GapToFastest_s", "FP2_LapCount", "FP2_Compounds"]])
    fp2_df.to_csv("monaco_fp2_pace.csv", index=False)
    print("\nSaved → monaco_fp2_pace.csv")

except Exception as e:
    print(f"FastF1 FP2 load error: {e}")
    print("Using manually sourced FP2 pace data from race-week reports...")
    # Manual FP2 data from formula1.com and racingnews365.com
    # HAM P1: 1:13.026, LEC +0.111, VER +0.168, RUS ~+0.300, ANT ~+0.350
    # PIA P7 from GPblog; BOR/HUL P8/P9 from GPblog; BEA P10 per GPblog
    # NOR ~P18 equivalent (only 8 laps, battery issue)
    fp2_manual = [
        {"Driver": "HAM", "FP2_GapToFastest_s": 0.000},
        {"Driver": "LEC", "FP2_GapToFastest_s": 0.111},
        {"Driver": "VER", "FP2_GapToFastest_s": 0.168},
        {"Driver": "RUS", "FP2_GapToFastest_s": 0.300},
        {"Driver": "ANT", "FP2_GapToFastest_s": 0.350},
        {"Driver": "HAD", "FP2_GapToFastest_s": 0.500},
        {"Driver": "PIA", "FP2_GapToFastest_s": 0.600},
        {"Driver": "HUL", "FP2_GapToFastest_s": 0.750},
        {"Driver": "BOR", "FP2_GapToFastest_s": 0.800},
        {"Driver": "BEA", "FP2_GapToFastest_s": 0.900},
        {"Driver": "LAW", "FP2_GapToFastest_s": 1.000},
        {"Driver": "GAS", "FP2_GapToFastest_s": 1.050},
        {"Driver": "COL", "FP2_GapToFastest_s": 1.150},
        {"Driver": "ALB", "FP2_GapToFastest_s": 1.200},
        {"Driver": "SAI", "FP2_GapToFastest_s": 1.250},
        {"Driver": "LIN", "FP2_GapToFastest_s": 1.350},
        {"Driver": "OCO", "FP2_GapToFastest_s": 1.500},
        {"Driver": "NOR", "FP2_GapToFastest_s": 1.800},  # limited laps (battery)
        {"Driver": "PER", "FP2_GapToFastest_s": 2.000},  # DNF in FP2 (smoke)
        {"Driver": "STR", "FP2_GapToFastest_s": 2.200},
        {"Driver": "ALO", "FP2_GapToFastest_s": 2.300},
        {"Driver": "BOT", "FP2_GapToFastest_s": 2.350},
    ]
    fp2_df = pd.DataFrame(fp2_manual)
    print(fp2_df)


# ── Merge all data ────────────────────────────────────────────────────────────
df = quali_df.merge(
    rolling_df[["Driver", "AvgFinishNorm", "RecentFinishNorm", "AvgGridNorm",
                "DNF_rate", "TeamScore", "Team"]],
    on="Driver", how="left"
)

df_final = df.merge(
    fp2_df[["Driver", "FP2_GapToFastest_s"]],
    on="Driver", how="left"
)


# ── Feature Engineering ───────────────────────────────────────────────────────
feature_cols = [
    "BestLap_s",             # Monaco quali: best lap
    "MonacoGapFromPole_s",   # Monaco quali: gap to pole
    "MonacoGrid",            # Monaco qualifying position
    "FP2_GapToFastest_s",    # FP2 long-run race pace (key at Monaco)
    "TeamScore",             # Constructor points normalised (after Canada R5)
    "DriverPointsNorm",      # Driver points normalised (after Canada R5)
    "AvgFinishNorm",         # Rolling avg finish (AUS+CHN+JPN+MIA+CAN)
    "RecentFinishNorm",      # Weighted recent form (CAN 40%, MIA 25%, JPN 20%, CHN 10%, AUS 5%)
    "AvgGridNorm",           # Rolling avg grid (5 races)
    "DNF_rate",              # Reliability signal (5 races)
    "RainProbability",       # Weather: ~0% rain — dry race expected
    "Temperature",           # Weather: ~23°C race day
]

# ── Target variable ───────────────────────────────────────────────────────────
# Monaco note: track position CRITICAL — grid position weighted more heavily
# Overtaking near-impossible → Monaco grid weight bumped to 0.25
df_final["RaceScore"] = (
    0.45 * df_final["AvgFinishNorm"] +
    0.30 * df_final["RecentFinishNorm"] +
    0.25 * df_final["AvgGridNorm"]
).round(4)

X = df_final[feature_cols].copy()
y = df_final["RaceScore"]

imputer   = SimpleImputer(strategy="median")
X_imputed = imputer.fit_transform(X)


# ── Multi-model LOO-CV comparison ─────────────────────────────────────────────
models = {
    "Ridge":         Ridge(alpha=1.0),
    "ElasticNet":    ElasticNet(alpha=0.1, l1_ratio=0.5),
    "RandomForest":  RandomForestRegressor(
                         n_estimators=100, max_depth=3,
                         min_samples_leaf=3, random_state=42),
    "GradientBoost": GradientBoostingRegressor(
                         n_estimators=100, learning_rate=0.05,
                         max_depth=2, subsample=0.8, random_state=42),
    "XGBoost":       XGBRegressor(
                         n_estimators=100, learning_rate=0.05,
                         max_depth=2, subsample=0.8,
                         colsample_bytree=0.8, reg_lambda=2.0,
                         random_state=42),
}

loo     = LeaveOneOut()
results = {}

for name, mdl in models.items():
    pipe = (Pipeline([("scaler", StandardScaler()), ("model", mdl)])
            if name in ["Ridge", "ElasticNet"]
            else Pipeline([("model", mdl)]))
    errors = []
    for train_idx, test_idx in loo.split(X_imputed):
        pipe.fit(X_imputed[train_idx], y.iloc[train_idx])
        errors.append(abs(pipe.predict(X_imputed[test_idx])[0] - y.iloc[test_idx].iloc[0]))
    results[name] = {"MAE": round(np.mean(errors), 4), "StdDev": round(np.std(errors), 4)}

print(f"\n{'Model':<18} {'LOO MAE':>8} {'Std Dev':>8}  {'Verdict'}")
print("─" * 55)
for name, res in sorted(results.items(), key=lambda x: x[1]["MAE"]):
    best = " ← best" if res["MAE"] == min(r["MAE"] for r in results.values()) else ""
    print(f"{name:<18} {res['MAE']:>8.4f} {res['StdDev']:>8.4f}  {best}")


# ── Best model final prediction ───────────────────────────────────────────────
best_name  = min(results, key=lambda x: results[x]["MAE"])
best_model = models[best_name]
final_pipe = (Pipeline([("scaler", StandardScaler()), ("model", best_model)])
              if best_name in ["Ridge", "ElasticNet"]
              else Pipeline([("model", best_model)]))
final_pipe.fit(X_imputed, y)
df_final["PredictedScore"] = final_pipe.predict(X_imputed)

final = df_final.sort_values("PredictedScore").reset_index(drop=True)

print("\n" + "=" * 72)
print("   PREDICTED FINISHING ORDER — 2026 MONACO GRAND PRIX")
print("   Rolling target: AUS + CHN + JPN + MIA + CAN avg finish + recent form")
print("=" * 72)
print(f"  {'P':<4}{'DRV':<6}{'Team':<22}{'Score'}")
print("  " + "─" * 48)
for i, row in final.iterrows():
    print(f"  {i+1:<4}{row['Driver']:<6}{row['Team']:<22}{row['PredictedScore']:.4f}")

podium = final.head(3)
print(f"\n  {'='*52}")
print(f"  🥇 P1: {podium.iloc[0]['Driver']} ({podium.iloc[0]['Team']})")
print(f"  🥈 P2: {podium.iloc[1]['Driver']} ({podium.iloc[1]['Team']})")
print(f"  🥉 P3: {podium.iloc[2]['Driver']} ({podium.iloc[2]['Team']})")
print(f"\n  Best model:   {best_name}")
print(f"  LOO MAE:      {results[best_name]['MAE']:.4f}")
print(f"  LOO Std Dev:  {results[best_name]['StdDev']:.4f}")
print(f"  Weather:      {df_final['Temperature'].iloc[0]:.1f}°C | "
      f"Rain: {df_final['RainProbability'].iloc[0]:.0%}")
print(f"  Monaco note:  Grid weight increased (0.25 vs 0.20 standard)")
print(f"                Overtaking near-impossible on 3.337km street circuit")
print(f"  {'='*52}")


# ── Feature importances / coefficients ───────────────────────────────────────
if best_name in ["RandomForest", "GradientBoost", "XGBoost"]:
    importances = final_pipe.named_steps["model"].feature_importances_
    print("\nFeature importances:")
    for feat, imp in sorted(zip(feature_cols, importances), key=lambda x: -x[1]):
        bar = "█" * int(imp * 50)
        print(f"  {feat:<32} {imp:.4f}  {bar}")
elif best_name in ["Ridge", "ElasticNet"]:
    coefficients = final_pipe.named_steps["model"].coef_
    print("\nFeature coefficients:")
    for feat, coef in sorted(zip(feature_cols, coefficients), key=lambda x: -abs(x[1])):
        bar = "█" * int(abs(coef) * 20)
        print(f"  {feat:<32} {coef:+.4f}  {bar}")


# ── All-models side-by-side comparison ───────────────────────────────────────
all_predictions = {}
for name, mdl in models.items():
    pipe = (Pipeline([("scaler", StandardScaler()), ("model", mdl)])
            if name in ["Ridge", "ElasticNet"]
            else Pipeline([("model", mdl)]))
    pipe.fit(X_imputed, y)
    ranked = pd.Series(pipe.predict(X_imputed),
                       index=df_final["Driver"]).rank(method="min").astype(int)
    all_predictions[name] = ranked

pred_df = pd.DataFrame(all_predictions)
pred_df["BestModelPos"] = pd.Series(
    final_pipe.predict(X_imputed), index=df_final["Driver"]
).rank(method="min").astype(int)
pred_df = pred_df.sort_values("BestModelPos")

print("\n" + "=" * 100)
print("   PREDICTED FINISHING ORDER — 2026 MONACO GRAND PRIX  (all models)")
print("=" * 100)
col_w  = 14
header = f"  {'DRV':<6}{'Team':<22}"
for name in models: header += f"{name:>{col_w}}"
header += f"  {'★ Best':>{col_w}}"
print(header)
print("  " + "─" * 92)
for driver in pred_df.index:
    team = df_final.loc[df_final["Driver"] == driver, "Team"].values[0]
    row  = f"  {driver:<6}{team:<22}"
    for name in models:
        row += f"  P{pred_df.loc[driver, name]:<{col_w-2}}"
    row += f"  P{pred_df.loc[driver, 'BestModelPos']:<{col_w-2}}"
    print(row)

# ── Consensus podium ──────────────────────────────────────────────────────────
best_order = pred_df.sort_values("BestModelPos")
p1, p2, p3 = best_order.index[0], best_order.index[1], best_order.index[2]
t1 = df_final.loc[df_final["Driver"] == p1, "Team"].values[0]
t2 = df_final.loc[df_final["Driver"] == p2, "Team"].values[0]
t3 = df_final.loc[df_final["Driver"] == p3, "Team"].values[0]

print(f"\n  {'='*62}")
print(f"  Best model: {best_name} (LOO MAE: {results[best_name]['MAE']:.4f})")
print(f"  🥇 P1: {p1} ({t1})")
print(f"  🥈 P2: {p2} ({t2})")
print(f"  🥉 P3: {p3} ({t3})")
print(f"  Weather:    {df_final['Temperature'].iloc[0]:.1f}°C | "
      f"Rain: {df_final['RainProbability'].iloc[0]:.0%}")
print(f"  {'='*62}")

print("\n  CONSENSUS PODIUM (across all models):")
print("  " + "─" * 40)
medals = ["🥇", "🥈", "🥉"]
for pos in [1, 2, 3]:
    pos_counts = {}
    for name in models:
        for d in pred_df[pred_df[name] == pos].index:
            pos_counts[d] = pos_counts.get(d, 0) + 1
    if pos_counts:
        top   = max(pos_counts, key=pos_counts.get)
        votes = pos_counts[top]
        print(f"  {medals[pos-1]} P{pos}: {top} — {votes}/{len(models)} models agree")
print(f"  {'='*62}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 136.0/136.0 kB 8.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.6/70.6 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 68.2 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.6/55.6 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 5.0 MB/s eta 0:00:00


req         WARNING 	DEFAULT CACHE ENABLED! (24.0 KB) /root/.cache/fastf1


   Driver  BestLap_s  MonacoGrid  MonacoGapFromPole_s  MonacoGapFromPolePct  \
0     ANT     72.051           1                0.000                0.0000   
1     VER     72.094           2                0.043                0.0597   
2     HAM     72.279           3                0.228                0.3164   
3     LEC     72.380           4                0.329                0.4566   
4     HAD     72.420           5                0.369                0.5121   
5     RUS     72.445           6                0.394                0.5468   
6     PIA     72.560           7                0.509                0.7064   
7     NOR     72.620           8                0.569                0.7897   
8     GAS     72.780           9                0.729                1.0118   
9     LAW     72.900          10                0.849                1.1783   
10    ALB     73.100          11                1.049                1.4559   
11    SAI     73.150          12                1.09

core           INFO 	Loading data for Monaco Grand Prix - Practice 2 [v3.8.3]
INFO:fastf1.fastf1.core:Loading data for Monaco Grand Prix - Practice 2 [v3.8.3]
req            INFO 	No cached data found for session_info. Loading data...
INFO:fastf1.fastf1.req:No cached data found for session_info. Loading data...
_api           INFO 	Fetching session info data...
INFO:fastf1.api:Fetching session info data...
DEBUG:fastf1.api:Falling back to livetiming mirror (https://livetiming-mirror.fastf1.dev)
logger      WARNING 	Failed to load session info data!
DEBUG:fastf1.fastf1.core:Traceback for failure in session info data
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/fastf1/logger.py", line 112, in __wrapped
    return func(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/fastf1/core.py", line 1452, in _load_session_info
    self._session_info = api.session_info(self.api_path,
                         ^^^^^^

FastF1 FP2 load error: The data you are trying to access has not been loaded yet. See `Session.load`
Using manually sourced FP2 pace data from race-week reports...
   Driver  FP2_GapToFastest_s
0     HAM               0.000
1     LEC               0.111
2     VER               0.168
3     RUS               0.300
4     ANT               0.350
5     HAD               0.500
6     PIA               0.600
7     HUL               0.750
8     BOR               0.800
9     BEA               0.900
10    LAW               1.000
11    GAS               1.050
12    COL               1.150
13    ALB               1.200
14    SAI               1.250
15    LIN               1.350
16    OCO               1.500
17    NOR               1.800
18    PER               2.000
19    STR               2.200
20    ALO               2.300
21    BOT               2.350

Model               LOO MAE  Std Dev  Verdict
───────────────────────────────────────────────────────
Ridge                0.0114   0.0108   ← be